<div align="center">

# <span style="color:#2E86C1;"> Why Optimizers Zigzag — and How Each Fix Introduces a New Flaw</span>

**Author:** [<span style="color:#8E44AD;">Dr. D Bhanu Prakash</span>](https://dbhanuprakash233.github.io)

<img src="https://github.com/dbhanuprakash233/SSSIHL_DBP/blob/main/assets/SssihlLogo.jpeg?raw=true" alt="University Logo" width="80"/>

**<span style="color:#16A085;">Sri Sathya Sai Institute of Higher Learning</span>**  
<span style="color:#5D6D7E;">Prasanthi Nilayam - 515 134, Andhra Pradesh, India.</span>

**Course:** <span style="color:#D35400;">Optimization Techniques for Machine Learning</span>  
**Course Code:** <span style="color:#1ABC9C;">UMAM-502</span>

</div>


# Gradient Descent: Interactive 3D/2D Demos

**Two things only, both fully interactive (drag to rotate, scroll to zoom, hover for values):**

1. **The zigzag pattern** — why gradient descent bounces back and forth instead of heading straight
   for the minimum.
2. **Choosing the step size (alpha)** — what happens when alpha is too high, too low, or "just right."

We use one running test problem throughout so the comparisons are fair:

$$f(x, y) = \tfrac12\left(a\,x^2 + b\,y^2\right), \qquad \nabla f(x,y) = (a x,\; b y), \qquad H = \begin{pmatrix}a & 0\\ 0 & b\end{pmatrix}$$

- $a, b$ are the curvatures (Hessian eigenvalues) along $x$ and $y$.
- **Condition number** $\kappa = b/a$ measures how "stretched" the bowl is.
- $\kappa = 1$ → circular bowl (easy). $\kappa \gg 1$ → elongated bowl (hard, causes zigzag).

Think of it as a valley: $x$ is the gently-sloped direction ($a=1$), $y$ is the steep direction
($b=50$). Rotate the 3D plots to see it's a **stretched, elongated valley** — not a round bowl.
That stretching is the whole reason for the zigzag.

In [ ]:

import numpy as np
import plotly.graph_objects as go
import plotly.offline as pyo
from plotly.subplots import make_subplots

pyo.init_notebook_mode(connected=False)   # embed plotly.js once so this notebook works offline

# ---- the valley -----------------------------------------------------
a, b = 1.0, 50.0                 # curvature along x (flat) and y (steep)
A = np.array([[a, 0.0], [0.0, b]])
kappa = b / a

def f(x, y):
    return 0.5 * (a * x**2 + b * y**2)

def grad(x, y):
    return np.array([a * x, b * y])

def gradient_descent(start, alpha, n_iter):
    x = np.array(start, dtype=float)
    path = [x.copy()]
    for _ in range(n_iter):
        x = x - alpha * grad(*x)
        path.append(x.copy())
    return np.array(path)

start = (8.0, 1.0)
print(f"Condition number kappa = b/a = {kappa:.0f}  (the valley is {kappa:.0f}x steeper in y than in x)")



### Helper functions for the interactive plots

(Just plotting machinery — feel free to skip reading this cell.)


In [ ]:

def surface_trace(xr, yr, n=80, colorscale='Blues', opacity=0.85):
    xs = np.linspace(*xr, n)
    ys = np.linspace(*yr, n)
    X, Y = np.meshgrid(xs, ys)
    Z = f(X, Y)
    return go.Surface(x=X, y=Y, z=Z, colorscale=colorscale, opacity=opacity,
                       showscale=False, name='f(x,y)',
                       contours={"z": {"show": True, "usecolormap": True,
                                       "highlightcolor": "limegreen", "project_z": False}})

def path_trace_3d(path, color='crimson', name='path'):
    zs = f(path[:, 0], path[:, 1])
    return go.Scatter3d(
        x=path[:, 0], y=path[:, 1], z=zs,
        mode='lines+markers',
        line=dict(color=color, width=5),
        marker=dict(size=4, color=list(range(len(path))), colorscale='Turbo', showscale=False),
        name=name,
        hovertemplate='step %{marker.color}<br>x=%{x:.3f}<br>y=%{y:.3f}<br>f=%{z:.3f}<extra></extra>',
    )

def make_3d_figure(path, xr, yr, title, camera=None):
    fig = go.Figure(data=[surface_trace(xr, yr), path_trace_3d(path)])
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='f(x, y)',
                    camera=camera or dict(eye=dict(x=1.6, y=-1.8, z=1.1))),
        width=780, height=560, margin=dict(l=10, r=10, t=50, b=10),
    )
    return fig

def contour_trace(xr, yr, n=200):
    xs = np.linspace(*xr, n)
    ys = np.linspace(*yr, n)
    X, Y = np.meshgrid(xs, ys)
    Z = f(X, Y)
    return go.Contour(x=xs, y=ys, z=Z, showscale=False,
                       colorscale='Greys', ncontours=35,
                       contours=dict(coloring='lines'), line_width=1,
                       hoverinfo='skip')

def path_trace_2d(path, color='crimson', name='path'):
    gx, gy = grad(path[:, 0], path[:, 1]).reshape(2, -1) if False else (None, None)
    losses = f(path[:, 0], path[:, 1])
    grads = np.array([grad(*p) for p in path])
    return go.Scatter(
        x=path[:, 0], y=path[:, 1], mode='lines+markers', name=name,
        line=dict(color=color, width=2),
        marker=dict(size=6, color=list(range(len(path))), colorscale='Turbo', showscale=False),
        customdata=np.column_stack([losses, grads[:, 0], grads[:, 1]]),
        hovertemplate=('step %{marker.color}<br>x=%{x:.3f}, y=%{y:.3f}<br>f=%{customdata[0]:.3f}'
                        '<br>grad=(%{customdata[1]:.2f}, %{customdata[2]:.2f})<extra></extra>'),
    )

def make_2d_figure(path, xr, yr, title):
    fig = go.Figure(data=[contour_trace(xr, yr), path_trace_2d(path)])
    fig.update_layout(title=title, xaxis_title='x', yaxis_title='y',
                       width=650, height=520, margin=dict(l=10, r=10, t=50, b=10))
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    return fig



## 1. The Zigzag Pattern

We take a fixed step size and run gradient descent on the elongated valley. Watch the path on the
**3D surface**: it keeps crossing from one wall of the valley to the other instead of running along
the valley floor toward the minimum.

**Why:** at every point, gradient descent moves *exactly perpendicular to the local contour line* (the
steepest way down *right where you're standing*). On a stretched valley, "steepest right now" and
"toward the minimum" point in very different directions — so the path overcorrects sideways every step.

**Try this:** drag the 3D plot to view the valley from above (top-down) — you'll see the zigzag path
looks like a lightning bolt, always crossing back and forth across the narrow direction.


In [ ]:

alpha_zigzag = 0.65 * (2 / b)      # comfortably stable, but still causes visible zigzag
path_zz = gradient_descent(start, alpha_zigzag, n_iter=22)

fig3d = make_3d_figure(path_zz, xr=(-9, 9), yr=(-2, 2),
                        title=f'Zigzag on an elongated valley (kappa={kappa:.0f}, alpha={alpha_zigzag:.3f})<br>'
                              f'<sub>Drag to rotate. Try looking straight down from above.</sub>')
fig3d.show()


In [ ]:

fig2d = make_2d_figure(path_zz, xr=(-9, 9), yr=(-2, 2),
                        title='Same path, top-down view (contour = lines of equal height)<br>'
                              '<sub>Hover any point to see its gradient vector</sub>')
fig2d.show()



### What to notice

- Every marker is one gradient-descent step; color goes from purple (start) to yellow (later steps).
- Hover over the early steps in the 2D plot — the gradient vector printed in the tooltip always points
  mostly along $y$ (the steep direction), because $|\partial f/\partial y| = b|y| \gg a|x| = |\partial f/\partial x|$
  even though $x$ is much farther from the minimum than $y$ is.
- That's the zigzag mechanism in one sentence: **the gradient is dominated by the steep direction, so
  the flat direction — where the minimum actually is — gets almost ignored.**

This is a property of the *shape of the bowl* (its condition number $\kappa = b/a$), not of the step
size. Which brings us to the next question: does picking a different step size fix it?



## 2. Choosing Alpha: Too High vs. Too Low vs. "Good"

Keep the same valley. Now compare three fixed step sizes:

- **Too high** — bigger than the stability limit for the steep direction → diverges (blows up).
- **Too low** — very safe, but tiny → converges, but painfully slowly.
- **"Good"** — the largest stable step size → converges fastest *of the three*, but **still zigzags**
  (as we just saw in Section 1).


In [ ]:

alpha_limit = 2 / b     # theoretical stability threshold for the steep axis

alpha_high = 1.15 * alpha_limit    # ABOVE the limit -> diverges
alpha_low  = 0.05 * alpha_limit    # tiny -> crawls
alpha_good = 0.9  * alpha_limit    # just under the limit -> fastest stable option

path_high = gradient_descent(start, alpha_high, n_iter=16)
path_low  = gradient_descent(start, alpha_low,  n_iter=150)
path_good = gradient_descent(start, alpha_good, n_iter=80)

print(f"Stability limit alpha < {alpha_limit:.4f}  (set by the STEEP direction, b={b})")
print(f"  Too high : alpha = {alpha_high:.4f}  -> diverges")
print(f"  Too low  : alpha = {alpha_low:.4f}  -> converges, needs {len(path_low)-1}+ steps")
print(f"  Good     : alpha = {alpha_good:.4f}  -> converges fastest, but still zigzags")


In [ ]:

# --- interactive 3D view, one dropdown to switch between the 3 cases ---
reach = np.abs(path_high).max() * 1.15
fig = go.Figure()

fig.add_trace(surface_trace((-reach, reach), (-reach, reach), n=70))
fig.add_trace(path_trace_3d(path_high, color='crimson', name='alpha too HIGH'))
fig.add_trace(path_trace_3d(path_low,  color='royalblue', name='alpha too LOW'))
fig.add_trace(path_trace_3d(path_good, color='seagreen', name='alpha GOOD'))

# traces: [0]=surface (always on), [1]=high, [2]=low, [3]=good
fig.update_traces(visible=True)
for i in (2, 3):
    fig.data[i].visible = False

buttons = []
labels = ['Too HIGH (diverges)', 'Too LOW (crawls)', "GOOD (fastest stable, still zigzags)"]
for i, label in enumerate(labels):
    vis = [True, False, False, False]
    vis[i + 1] = True
    buttons.append(dict(label=label, method='update', args=[{'visible': vis}]))

fig.update_layout(
    title='Click a button to compare alpha choices (same valley, same start point)',
    updatemenus=[dict(type='buttons', direction='right', x=0.5, xanchor='center',
                       y=1.15, showactive=True, buttons=buttons)],
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='f(x, y)',
               camera=dict(eye=dict(x=1.6, y=-1.8, z=1.1))),
    width=800, height=600, margin=dict(l=10, r=10, t=90, b=10),
)
fig.show()


In [ ]:

# --- interactive loss-vs-iteration comparison (log scale, zoomable, with range slider) ---
loss_high = f(path_high[:, 0], path_high[:, 1])
loss_low  = f(path_low[:, 0],  path_low[:, 1])
loss_good = f(path_good[:, 0], path_good[:, 1])

fig = go.Figure()
fig.add_trace(go.Scatter(y=loss_high, mode='lines+markers', name=f'Too HIGH (alpha={alpha_high:.4f})', line=dict(color='crimson')))
fig.add_trace(go.Scatter(y=loss_low,  mode='lines+markers', name=f'Too LOW (alpha={alpha_low:.4f})', line=dict(color='royalblue')))
fig.add_trace(go.Scatter(y=loss_good, mode='lines+markers', name=f'GOOD (alpha={alpha_good:.4f})', line=dict(color='seagreen')))

fig.update_layout(
    title='f(x, y) vs. iteration — drag the range slider below to zoom in',
    xaxis_title='iteration', yaxis_title='f(x, y)',
    yaxis_type='log',
    xaxis=dict(rangeslider=dict(visible=True)),
    width=800, height=480, margin=dict(l=10, r=10, t=50, b=10),
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig.show()



### What to notice

- **Too high** (red): the loss *increases* — the surface plot shows the path shooting out of the
  valley entirely, and the log-scale loss curve rockets upward instead of down.
- **Too low** (blue): completely stable, but the loss barely drops even after 150 steps — zoom into
  the range slider to see just how flat that blue line is.
- **Good** (green): converges fastest of the three, but if you switch back to the 3D view you can see
  it's still visibly bouncing side to side while it descends — the zigzag from Section 1 doesn't go
  away just because alpha is well chosen. Alpha only controls **how fast you crawl along this zigzag
  path** — it can't change the *shape* of the path, because that shape is set by the valley's condition
  number $\kappa = b/a$, not by alpha.

**One-line takeaway:** alpha has a narrow safe range (too high diverges, too low crawls), and even the
best alpha in that range can't undo the zigzag — fixing *that* requires changing the search *direction*
itself (e.g. Newton's method, momentum, or adaptive methods), not just the step size.


**A guided tour: Steepest Descent → Gradient Descent → Stochastic Gradient Descent → Newton's Method → BFGS (Quasi-Newton)**

Each section below follows the same pattern:

1. **Try the method.**
2. **See exactly why it struggles** (good example vs. bad example).
3. **Motivate the next method** as a fix for that specific flaw.
4. **Show the new method has its own flaw**, which motivates the next section.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# ----- shared test problem -----------------------------------------
def make_problem(a, b):
    A = np.diag([a, b]).astype(float)
    def f(x):
        return 0.5 * x @ A @ x
    def grad(x):
        return A @ x
    def hess(x):
        return A
    return f, grad, hess, A

start = np.array([8.0, 1.0])

# "GOOD" problem: nearly circular bowl -> low condition number
a_good, b_good = 10.0, 12.0
f_good, grad_good, hess_good, A_good = make_problem(a_good, b_good)
kappa_good = b_good / a_good

# "BAD" problem: elongated bowl -> high condition number
a_bad, b_bad = 1.0, 50.0
f_bad, grad_bad, hess_bad, A_bad = make_problem(a_bad, b_bad)
kappa_bad = b_bad / a_bad

print(f"GOOD problem condition number kappa = {kappa_good:.1f}  (nearly circular)")
print(f"BAD  problem condition number kappa = {kappa_bad:.1f}  (elongated)")

def plot_path(ax, f, path, title, xr=(-9, 9), yr=(-2, 2), color='crimson', label=None):
    xs = np.linspace(*xr, 300)
    ys = np.linspace(*yr, 300)
    X, Y = np.meshgrid(xs, ys)
    Z = np.array([[f(np.array([xx, yy])) for xx in xs] for yy in ys])
    ax.contour(X, Y, Z, levels=25, cmap='Greys', linewidths=0.6)
    path = np.array(path)
    ax.plot(path[:, 0], path[:, 1], 'o-', color=color, ms=3, lw=1, label=label)
    ax.plot(0, 0, 'k*', ms=14)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    if label:
        ax.legend(fontsize=8, loc='upper right')



## 1. Steepest Descent (exact line search)

**Idea:** at every step, move along $-\nabla f(x)$, but instead of guessing a step size, solve *exactly* how far
to go — the optimal step size for a quadratic has a closed form:

$$\alpha_k = \frac{g_k^T g_k}{g_k^T A g_k}, \qquad x_{k+1} = x_k - \alpha_k g_k$$

This is the *best possible* step size at every single iteration. If step-size tuning were the problem,
this method should fix it completely.

### Good example vs. bad example


In [ ]:

def steepest_descent_exact(grad, A, x0, n_iter):
    x = x0.copy()
    path = [x.copy()]
    for _ in range(n_iter):
        g = grad(x)
        alpha = (g @ g) / (g @ A @ g)
        x = x - alpha * g
        path.append(x.copy())
    return np.array(path)

path_sd_good = steepest_descent_exact(grad_good, A_good, start, 20)
path_sd_bad  = steepest_descent_exact(grad_bad,  A_bad,  start, 40)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_path(axes[0], f_good, path_sd_good,
          f"GOOD: kappa={kappa_good:.1f} -> converges almost immediately\n"
          f"({len(path_sd_good)-1} steps shown)")
plot_path(axes[1], f_bad, path_sd_bad,
          f"BAD: kappa={kappa_bad:.1f} -> STILL zigzags, even with the\n"
          f"*optimal* step size every iteration! ({len(path_sd_bad)-1} steps)")
plt.tight_layout(); plt.show()



### What this proves

Even with a **perfectly tuned step size at every iteration**, the ill-conditioned problem still zigzags.

**Why:** the search direction $-\nabla f(x)$ is always perpendicular to the local contour line. On an
elongated bowl, "perpendicular to the contour" rarely points at the minimum — so no matter how well you
choose *how far* to step, the *direction* itself is bad.

> **Conclusion:** step-size tuning cannot fix zigzagging caused by ill-conditioning. This rules out
> "just use exact line search" as a solution — so ordinary (fixed-step) **Gradient Descent** must deal
> with this same directional problem, *plus* the extra headache of choosing a step size without the
> luxury of an exact line search.



## 2. Gradient Descent (fixed step size $\alpha$)

In practice we can't afford an exact line search every step (it requires knowing $A$!). Instead we pick
one fixed $\alpha$ and hope for the best. Let's see what happens for **too high**, **too low**, and
**"just right"** — all on the *same* bad (ill-conditioned) problem.


In [ ]:

def gradient_descent(grad, x0, alpha, n_iter):
    x = x0.copy()
    path = [x.copy()]
    for _ in range(n_iter):
        x = x - alpha * grad(x)
        path.append(x.copy())
    return np.array(path)

alpha_max_stable = 2 / b_bad          # theoretical stability limit for the steep axis
alphas = {
    "Too HIGH (unstable)": alpha_max_stable * 1.05,
    "Too LOW (crawls)":    alpha_max_stable * 0.05,
    "'Best' fixed alpha":  alpha_max_stable * 0.95,
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
paths_gd = {}
for ax, (label, alpha) in zip(axes, alphas.items()):
    n_iter = 15 if "HIGH" in label else 80
    path = gradient_descent(grad_bad, start, alpha, n_iter)
    paths_gd[label] = path
    plot_path(ax, f_bad, path, f"{label}\nalpha={alpha:.4f}", xr=(-40, 40), yr=(-8, 8))
plt.tight_layout(); plt.show()

print(f"Stability limit for the STEEP axis (b={b_bad}): alpha < {2/b_bad:.4f}")
print(f"Stability limit for the FLAT  axis (a={a_bad}): alpha < {2/a_bad:.4f}")
print("-> A single alpha must satisfy the tightest (steep-axis) constraint,")
print("   which makes progress along the flat axis unnecessarily slow.")



### What this proves

- **Too high** → diverges / explodes along the steep axis.
- **Too low** → stable, but painfully slow along the flat axis.
- **"Best" fixed $\alpha$** (just under the stability limit) → still zigzags, still slow — because
  it's a compromise dictated by the *stiffest* direction, and simply can't move faster along the *flat*
  direction without blowing up the *steep* one.

> **Conclusion:** Gradient Descent inherits the directional problem from Section 1, and *adds* a brittle
> step-size tuning problem on top. It also recomputes the **full gradient over the entire dataset** at
> every step — for large datasets ($n$ = millions of points) that's extremely expensive. This motivates
> **Stochastic Gradient Descent**, which fixes the *computational cost* problem.



## 3. Stochastic Gradient Descent (SGD)

Real ML problems are sums over many data points: $f(x) = \frac1N\sum_{i=1}^N f_i(x)$. Full-batch GD needs
*all* $N$ terms every step. **SGD** instead estimates the gradient using a small random sample (or a
single point) — much cheaper per step.

We simulate this by adding per-sample noise to our quadratic problem (like noisy least-squares data),
so each step only "sees" one noisy point instead of the true gradient.


In [ ]:

# Simulate N noisy linear-regression-style samples whose *average* gradient
# equals our ill-conditioned quadratic's true gradient.
N = 200
noise_scale = np.array([3.0, 15.0])           # noise scaled per-axis like the curvature
noise = np.random.randn(N, 2) * noise_scale

def stochastic_grad(x, i):
    # Gradient estimate using ONE noisy sample i (cheap!) instead of the true gradient.
    return A_bad @ x + noise[i]

def sgd(x0, alpha, n_iter, batch=1, decay=False):
    # Uses ONE noisy sample per step (cheap!) instead of the true gradient.
    x = x0.copy()
    path = [x.copy()]
    for k in range(n_iter):
        idx = np.random.choice(N, size=batch, replace=False)
        g_est = np.mean([stochastic_grad(x, i) for i in idx], axis=0)
        a_k = alpha / (1 + 0.01 * k) if decay else alpha   # optional decaying LR
        x = x - a_k * g_est
        path.append(x.copy())
    return np.array(path)

alpha_sgd = 2 / b_bad * 0.6
path_sgd_fixed = sgd(start, alpha_sgd, 300, batch=1, decay=False)
path_sgd_decay = sgd(start, alpha_sgd, 300, batch=1, decay=True)
path_gd_full   = gradient_descent(grad_bad, start, alpha_sgd, 300)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_path(axes[0], f_bad, path_gd_full,
          "Full-batch GD (baseline)\nsmooth path, uses ALL N points every step",
          xr=(-9, 9), yr=(-2, 2))
plot_path(axes[1], f_bad, path_sgd_fixed,
          "SGD, fixed alpha\nnoisy: bounces around near the minimum, never fully settles",
          xr=(-9, 9), yr=(-2, 2), color='darkorange')
plot_path(axes[2], f_bad, path_sgd_decay,
          "SGD, DECAYING alpha\nnoise shrinks over time -> settles down",
          xr=(-9, 9), yr=(-2, 2), color='seagreen')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy([f_bad(p) for p in path_gd_full],   label='Full-batch GD', color='crimson')
ax.semilogy([f_bad(p) for p in path_sgd_fixed], label='SGD (fixed alpha)', color='darkorange')
ax.semilogy([f_bad(p) for p in path_sgd_decay], label='SGD (decaying alpha)', color='seagreen')
ax.set_xlabel('iteration'); ax.set_ylabel('f(x,y) [log]')
ax.set_title('SGD is cheap per step, but noisy: it never fully converges\nunless the learning rate decays')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()



### What this proves

- SGD is **much cheaper per step** (1 sample instead of $N$) — this fixes GD's computational-cost flaw.
- But the noisy gradient estimate means the path **never settles exactly at the minimum** with a fixed
  $\alpha$ — it bounces around in a noise "ball." A decaying learning rate helps, but is yet another
  hyperparameter to tune, and convergence is still slow along the flat direction (same ill-conditioning
  problem as before, now *compounded* by noise).

> **Conclusion:** SGD solves the *cost* problem but not the *conditioning* problem, and adds a *noise*
> problem. All three methods so far (Steepest Descent, GD, SGD) only ever use **first-order information**
> (the gradient). They have no idea the bowl is stretched — they can't "see" curvature. **Newton's Method**
> fixes this by using **second-order information** (the Hessian) directly.



## 4. Newton's Method

**Idea:** instead of stepping along $-\nabla f(x)$, *rescale* the step by the inverse Hessian:

$$x_{k+1} = x_k - H^{-1}\nabla f(x_k)$$

Geometrically, multiplying by $H^{-1}$ **undoes the stretching of the bowl** — it turns the elongated
ellipse back into a circle before stepping. For a quadratic, $H$ is constant and exact, so this should
converge **in a single step**, regardless of how ill-conditioned the problem is.


In [ ]:

def newton(grad, hess, x0, n_iter):
    x = x0.copy()
    path = [x.copy()]
    for _ in range(n_iter):
        g = grad(x)
        H = hess(x)
        x = x - np.linalg.solve(H, g)     # x - H^{-1} g, without forming H^{-1} explicitly
        path.append(x.copy())
        if np.linalg.norm(g) < 1e-10:
            break
    return np.array(path)

path_newton = newton(grad_bad, hess_bad, start, 5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_path(axes[0], f_bad, path_sd_bad,
          f"Steepest Descent (best case from Sec.1)\n{len(path_sd_bad)-1} steps, still zigzags")
plot_path(axes[1], f_bad, path_newton,
          f"Newton's Method\n{len(path_newton)-1} step(s) -> DONE. No zigzag at all.",
          color='seagreen')
plt.tight_layout(); plt.show()

print(f"Newton's method reached the minimum in {len(path_newton)-1} step(s),")
print(f"regardless of kappa = {kappa_bad:.0f}, because H^-1 exactly cancels the curvature.")



### What this proves

Newton's method is **immune to ill-conditioning** for quadratics — it converges in one step because
$H^{-1}$ perfectly corrects for the stretched geometry. This directly fixes the root cause identified
back in Section 1.

### ...but Newton's method has a serious flaw

- Computing and inverting (or solving a linear system with) the Hessian costs roughly $O(n^3)$ for $n$
  parameters. For a deep learning model with millions/billions of parameters, forming an $n\times n$
  Hessian is **completely infeasible**.
- The Hessian must also be positive definite for the step to point "downhill" — on non-convex problems
  (e.g. near saddle points) plain Newton steps can move *uphill* or diverge.

> **Conclusion:** Newton's method is the ideal *direction*, but computing it exactly is too expensive.
> This motivates methods that **approximate** the curvature information cheaply using only gradients —
> **Quasi-Newton methods**, the most famous of which is **BFGS**.



## 5. BFGS (Quasi-Newton) — approximating curvature without the Hessian

**Idea:** build up an approximation $B_k \approx H^{-1}$ using only the gradients seen so far (no second
derivatives needed!). Each step updates the curvature estimate using the change in gradient between
consecutive iterations (the *secant equation*), then takes a Newton-like step with the *approximate*
inverse Hessian:

$$x_{k+1} = x_k - \alpha_k B_k \nabla f(x_k)$$

This gets most of Newton's benefit (correcting for curvature) at a fraction of the cost (no $O(n^3)$
Hessian formation — BFGS updates cost $O(n^2)$, and limited-memory variants like L-BFGS cost $O(n)$,
which is why L-BFGS is standard for large-scale problems).


In [ ]:

def bfgs(grad, x0, n_iter, tol=1e-10):
    x = x0.copy()
    n = len(x)
    B = np.eye(n)                      # initial guess: no curvature info yet (behaves like GD at first)
    g = grad(x)
    path = [x.copy()]
    for _ in range(n_iter):
        if np.linalg.norm(g) < tol:
            break
        p = -B @ g                                  # quasi-Newton direction
        # simple backtracking line search for stability
        alpha = 1.0
        while f_bad(x + alpha * p) > f_bad(x) + 1e-4 * alpha * g @ p and alpha > 1e-10:
            alpha *= 0.5
        x_new = x + alpha * p
        g_new = grad(x_new)

        s = x_new - x
        y = g_new - g
        sy = s @ y
        if sy > 1e-12:                               # BFGS inverse-Hessian update
            rho = 1.0 / sy
            I = np.eye(n)
            B = (I - rho * np.outer(s, y)) @ B @ (I - rho * np.outer(y, s)) + rho * np.outer(s, s)

        x, g = x_new, g_new
        path.append(x.copy())
    return np.array(path)

path_bfgs = bfgs(grad_bad, start, 30)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_path(axes[0], f_bad, path_newton,
          f"Newton's Method\n{len(path_newton)-1} step, needs the TRUE Hessian (O(n^3) cost)",
          color='seagreen')
plot_path(axes[1], f_bad, path_bfgs,
          f"BFGS (Quasi-Newton)\n{len(path_bfgs)-1} steps, NO Hessian needed - only gradients",
          color='mediumpurple')
plt.tight_layout(); plt.show()



### What this proves

BFGS reaches the minimum in a handful of steps (far fewer than Steepest Descent / GD / SGD), gets close
to Newton's efficiency, and **never explicitly forms the Hessian** — it only ever uses gradient
evaluations, so each step is cheap. This is why BFGS / L-BFGS is the workhorse of classical
(non-stochastic) numerical optimization.

**BFGS's own flaw:** it still assumes a smooth, well-behaved deterministic gradient — it doesn't mix well
with the *noisy* mini-batch gradients from Section 3 (its curvature estimate gets corrupted by noise). This
is why large-scale deep learning uses neither pure Newton nor pure BFGS, but **stochastic** adaptive
methods (Adam, RMSProp, etc.) that blend ideas from *all* the sections above: cheap noisy gradients (SGD)
+ momentum/adaptive per-direction scaling (a cheap stand-in for curvature, like BFGS) — a good "extra
credit" experiment for you to try next!



## Final Comparison — All Methods Side-by-Side


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5.5))
curves = {
    "Steepest Descent (exact line search)": ([f_bad(p) for p in path_sd_bad], 'crimson'),
    "Gradient Descent ('best' fixed alpha)": ([f_bad(p) for p in paths_gd["'Best' fixed alpha"]], 'darkorange'),
    "SGD (decaying alpha)":                  ([f_bad(p) for p in path_sgd_decay], 'goldenrod'),
    "Newton's Method":                       ([f_bad(p) for p in path_newton], 'seagreen'),
    "BFGS (Quasi-Newton)":                   ([f_bad(p) for p in path_bfgs], 'mediumpurple'),
}
for label, (loss, color) in curves.items():
    ax.semilogy(loss, label=f"{label}  ({len(loss)-1} steps)", color=color)
ax.set_xlabel('iteration'); ax.set_ylabel('f(x,y)  [log scale]')
ax.set_title(f'All methods on the SAME ill-conditioned problem (kappa={kappa_bad:.0f})')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()



### Summary table

| Method | Fixes | Introduces |
|---|---|---|
| **Steepest Descent** (exact line search) | Removes step-size guesswork | Still zigzags — direction itself is bad on ill-conditioned problems |
| **Gradient Descent** (fixed $\alpha$) | Cheap, no line search needed | Must tune $\alpha$; a single $\alpha$ can't serve both flat & steep directions |
| **Stochastic Gradient Descent** | Cheap *per step* even for huge datasets | Noisy gradient estimates → never fully settles without LR decay |
| **Newton's Method** | Uses curvature ($H^{-1}$) → converges in 1 step, no zigzag | $O(n^3)$ Hessian cost; unreliable if $H$ isn't positive definite |
| **BFGS (Quasi-Newton)** | Approximates curvature from gradients only, no explicit Hessian | Curvature estimate breaks down under stochastic/noisy gradients |

**The big picture:** every method here is trying to answer the same question — *"how do I combine
gradient information with an estimate of curvature, cheaply and reliably?"* Steepest Descent and GD
ignore curvature entirely (that's the zigzag). SGD makes gradients cheap but noisy. Newton uses curvature
perfectly but expensively. BFGS approximates curvature cheaply but assumes clean gradients. Modern deep
learning optimizers (Adam, RMSProp) are the next chapter in this same story.
